In [ ]:
!pip install evaluate bert_score transformers rouge_score scispacy
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

import pandas as pd
import numpy as np
import re
import evaluate
import spacy

from google.colab import drive
drive.mount('/content/drive')

#Cargamos los datos
ruta_csv = '/content/drive/MyDrive/Bioinformática_LLM/Temp05/resultados_modelo_finetuned_complejos_temp05.csv'
df = pd.read_csv(ruta_csv)

columna_respuesta = 'Response_FINETUNED_Model'  # o 'Response_FINETUNED_Model'
df[columna_respuesta] = df[columna_respuesta].fillna("").astype(str)

referencias = df['Real_Diagnosis'].tolist()
predicciones = df[columna_respuesta].tolist()

# ================================================================
#  1. BERTScore (F1)
# ================================================================
bertscore_metric = evaluate.load("bertscore")
bert_results = bertscore_metric.compute(
    predictions=predicciones,
    references=referencias,
    lang="en",
    model_type="roberta-base"
)
df['Metrica_BERTScore_F1'] = bert_results['f1']

# ================================================================
#  2. ACIERTO POR PALABRAS
# ================================================================
mapeo_palabras_clave = {
    "Systemic Lupus Erythematosus (SLE)": ["lupus", "sle"],
    "Pheochromocytoma": ["pheochromocytoma"],
    "Infective Endocarditis": ["endocarditis"],
    "Myasthenia Gravis": ["myasthenia"],
    "Pulmonary Thromboembolism (PTE)": ["thromboembolism", "pte", "pulmonary embolism"],
    "Lyme Disease": ["lyme", "borreliosis"],
    "Multiple Sclerosis": ["multiple sclerosis"],
    "Addison's Disease": ["addison", "adrenal insufficiency"],
    "Ruptured Ectopic Pregnancy": ["ectopic"],
    "Guillain-Barré Syndrome": ["guillain", "barré", "barre"]
}

def verificar_acierto(row):
    enfermedad = row['Real_Diagnosis']
    respuesta = row[columna_respuesta].lower()
    palabras_validas = mapeo_palabras_clave.get(enfermedad, [])
    return 1 if any(palabra in respuesta for palabra in palabras_validas) else 0

df['Metrica_Acierto_PalabrasClave'] = df.apply(verificar_acierto, axis=1)

# ================================================================
#  3. ROUGE-L (F1)
# ================================================================

rouge_metric = evaluate.load("rouge")
rouge_results = rouge_metric.compute(
    predictions=predicciones,
    references=referencias,
    rouge_types=["rougeL"]
)
df['Metrica_ROUGE_L_F1'] = rouge_results['rougeL']

# ================================================================
#  4. ACIERTO ESTRUCTURADO
# ================================================================
normalizacion = {
    "Systemic Lupus Erythematosus (SLE)": "systemic lupus erythematosus",
    "Pheochromocytoma": "pheochromocytoma",
    "Infective Endocarditis": "infective endocarditis",
    "Myasthenia Gravis": "myasthenia gravis",
    "Pulmonary Thromboembolism (PTE)": "pulmonary thromboembolism",
    "Lyme Disease": "lyme disease",
    "Multiple Sclerosis": "multiple sclerosis",
    "Addison's Disease": "addison's disease",
    "Ruptured Ectopic Pregnancy": "ruptured ectopic pregnancy",
    "Guillain-Barré Syndrome": "guillain-barré syndrome"
}

def extraer_diagnostico(texto):
    inicio_pat = r"(?:^|\n)\s*(?:\d+\.\s*)?\**Main\s+diagnostic\s+suspicion\s*:?\**"
    inicio_match = re.search(inicio_pat, texto, re.IGNORECASE)
    if not inicio_match:
        return ""
    start_pos = inicio_match.end()
    siguiente_pat = r"\n\s*(?:\d+\.\s*)?\**Brief\s+medical\s+justification"
    siguiente_match = re.search(siguiente_pat, texto[start_pos:], re.IGNORECASE)
    if siguiente_match:
        end_pos = start_pos + siguiente_match.start()
    else:
        end_pos = len(texto)
    diag_texto = texto[start_pos:end_pos].strip()
    diag_texto = re.sub(r'\*+', '', diag_texto).strip().rstrip('.').lower()
    return diag_texto

def verificar_acierto_estructurado(row):
    diag_extraido = extraer_diagnostico(row[columna_respuesta])
    canon_real = normalizacion.get(row['Real_Diagnosis'], row['Real_Diagnosis'].lower())
    return 1 if canon_real in diag_extraido else 0

df['Metrica_Acierto_Estructurado'] = df.apply(verificar_acierto_estructurado, axis=1)

# ================================================================
#  5. FORMATO CORRECTO
# ================================================================
def cumple_formato(texto):
    tiene_sospecha = bool(re.search(r"main\s+diagnostic\s+suspicion", texto, re.IGNORECASE))
    tiene_justificacion = bool(re.search(r"(?:brief\s+)?medical\s+justification", texto, re.IGNORECASE))
    return 1 if (tiene_sospecha and tiene_justificacion) else 0

df['Metrica_Formato_Correcto'] = df[columna_respuesta].apply(cumple_formato)

# ================================================================
#  6. MÉTRICAS DE ALUCINACIÓN
# ================================================================

tasa_alucinacion_principal = 100 - df['Metrica_Acierto_Estructurado'].mean() * 100
nlp = spacy.load("en_ner_bc5cdr_md")

# Diccionario de enfermedades reales
enfermedades_reales_ingles = {
    # Las 10 del dataset
    "systemic lupus erythematosus", "sle", "lupus",
    "pheochromocytoma",
    "infective endocarditis", "endocarditis",
    "myasthenia gravis", "myasthenia",
    "pulmonary thromboembolism", "pte", "pulmonary embolism",
    "lyme disease", "lyme", "borreliosis",
    "multiple sclerosis", "ms",
    "addison's disease", "addison", "adrenal insufficiency",
    "ruptured ectopic pregnancy", "ectopic pregnancy",
    "guillain-barré syndrome", "guillain-barre syndrome", "guillain barre",

    # Enfermedades que el modelo alucina con frecuencia
    "sjögren's syndrome", "sjogren syndrome", "sjögren",
    "sarcoidosis",
    "osteomyelitis",
    "rheumatoid arthritis",
    "fibromyalgia",
    "chronic fatigue syndrome",
    "discoid lupus",
    "antiphospholipid syndrome",
    "scleroderma",
    "systemic sclerosis",
    "polymyositis",
    "dermatomyositis",
    "mixed connective tissue disease",
    "candidiasis", "candida",
    "hypertensive crisis",
    "takotsubo syndrome",
    "stroke", "ischemic stroke",
    "transient ischemic attack",
    "migraine",
    "cluster headache",
    "anxiety disorder",
    "panic attack",
    "hyperthyroidism",
    "hypothyroidism",
    "diabetes mellitus",
    "hypertension",
    "pneumonia",
    "bronchitis",
    "asthma",
    "copd",
    "pulmonary edema",
    "heart failure",
    "myocardial infarction",
    "pericarditis",
    "aortic dissection",
    "deep vein thrombosis",
    "cellulitis",
    "abscess",
    "sepsis",
    "meningitis",
    "encephalitis",
    "neuropathy",
    "radiculopathy",
    "optic neuritis",
    "multiple system atrophy",
    "parkinson's disease",
    "alzheimer's disease",
    "huntington's disease",
    "amyotrophic lateral sclerosis",
    "vasculitis",
    "giant cell arteritis",
    "polymyalgia rheumatica",
    "crohn's disease",
    "ulcerative colitis",
    "celiac disease",
    "irritable bowel syndrome",
    "pancreatitis",
    "cholecystitis",
    "appendicitis",
    "diverticulitis",
    "endometriosis",
    "pelvic inflammatory disease",
    "ovarian cyst",
    "testicular torsion",
    "prostatitis",
    "urinary tract infection",
    "pyelonephritis",
    "kidney stone",
    "glomerulonephritis",
    "liver cirrhosis",
    "hepatitis",
    "pancreatic cancer",
    "lung cancer",
    "breast cancer",
    "colorectal cancer",
    "leukemia",
    "lymphoma",
    "melanoma",
    "squamous cell carcinoma",
    "basal cell carcinoma",
}

def contar_enfermedades_alternativas(row):
    enfermedad_real = row['Real_Diagnosis']
    respuesta = row[columna_respuesta]

    doc = nlp(respuesta)
    enfermedades_mencionadas = set()

    for ent in doc.ents:
        if ent.label_ == "DISEASE":
            nombre = ent.text.strip().lower()
            nombre = re.sub(r'[^\w\s]', '', nombre)
            if nombre and nombre in enfermedades_reales_ingles:
                enfermedades_mencionadas.add(nombre)

    #Normalizamos el diagnóstico real y sus sinónimos
    canon_real = normalizacion.get(enfermedad_real, enfermedad_real.lower())
    sinonimos = set()
    for palabra in mapeo_palabras_clave.get(enfermedad_real, []):
        sinonimos.add(palabra.lower())

    alternativas = set()
    for enf in enfermedades_mencionadas:
        if enf == canon_real or any(sin in enf for sin in sinonimos):
            continue
        alternativas.add(enf)

    return len(alternativas)

df['Metrica_Num_Alt_Diagnosticos'] = df.apply(contar_enfermedades_alternativas, axis=1)

media_alt_diag = df['Metrica_Num_Alt_Diagnosticos'].mean()
pct_con_alucinacion = (df['Metrica_Num_Alt_Diagnosticos'] > 0).mean() * 100

#Mostramos los resultados obtenidos
print("\n" + "="*50)
print("=== GLOBAL METRICS RESULTS ===")
print(f"1. BERTScore (Semantic) F1 mean    : {df['Metrica_BERTScore_F1'].mean():.4f}")
print(f"2. Keyword Match Accuracy          : {df['Metrica_Acierto_PalabrasClave'].mean()*100:.2f}%")
print(f"3. ROUGE-L F1 mean                 : {df['Metrica_ROUGE_L_F1'].mean():.4f}")
print(f"4. Structured Accuracy             : {df['Metrica_Acierto_Estructurado'].mean()*100:.2f}%")
print(f"5. Correct Format Rate             : {df['Metrica_Formato_Correcto'].mean()*100:.2f}%")
print(f"6. Diagnostic Hallucination Rate   : {tasa_alucinacion_principal:.2f}%")
print(f"7. Avg. alternative diagnoses      : {media_alt_diag:.2f}")
print(f"8. % responses with ≥1 alt. diag.  : {pct_con_alucinacion:.2f}%")
print("="*50)

#Guardamos las métricas calculadas en otro csv
ruta_salida = '/content/drive/MyDrive/Bioinformática_LLM/metricas_modelo_finetuned_completo_temp05.csv'
df.to_csv(ruta_salida, index=False)

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz (119.8 MB)
  Preparing metadata (setup.py) ... done
Mounted at /content/drive
Calculando BERTScore (English)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Calculando ROUGE-L...


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]



=== GLOBAL METRICS RESULTS ===
1. BERTScore (Semantic) F1 mean    : 0.8239
2. Keyword Match Accuracy          : 84.00%
3. ROUGE-L F1 mean                 : 0.0311
4. Structured Accuracy             : 33.00%
5. Correct Format Rate             : 54.00%
6. Diagnostic Hallucination Rate   : 67.00%
7. Avg. alternative diagnoses      : 0.34
8. % responses with ≥1 alt. diag.  : 30.00%

✅ Metrics CSV saved to: /content/drive/MyDrive/Bioinformática_LLM/metricas_modelo_finetuned_completo_temp05.csv
